Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Dataset

In [ ]:
df = pd.read_csv("/email spam detection dataset.zip", encoding='latin-1')

EDA

In [ ]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
df.shape

(5572, 5)

In [ ]:
df = df[['v1', 'v2']]

df.columns = ['label', 'message']

print(df.head())

   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


Encoding

In [ ]:
encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['label'])

/tmp/ipykernel_2567/450787652.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = encoder.fit_transform(df['label'])


Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['message'],
    df['label'],
    test_size=0.2,
    random_state=42
)

Tokenization

In [ ]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

Padding

In [ ]:
X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

Deep Learning Model

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))

model.add(LSTM(64))

model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model Compile

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

Training Model

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - accuracy: 0.9462 - loss: 0.1726 - val_accuracy: 0.9803 - val_loss: 0.0719
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step - accuracy: 0.9897 - loss: 0.0381 - val_accuracy: 0.9830 - val_loss: 0.0591
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 11s 58ms/step - accuracy: 0.9955 - loss: 0.0169 - val_accuracy: 0.9821 - val_loss: 0.0521
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 11s 66ms/step - accuracy: 0.9991 - loss: 0.0050 - val_accuracy: 0.9874 - val_loss: 0.0517
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 49ms/step - accuracy: 0.9998 - loss: 0.0019 - val_accuracy: 0.9830 - val_loss: 0.0688


Evaluation

In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Accuracy:", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9830 - loss: 0.0688
Accuracy: 0.9829596281051636


Predict New Message

In [ ]:
sample = ["Congratulations! You won a free iPhone"]

sample_seq = tokenizer.texts_to_sequences(sample)

sample_pad = pad_sequences(sample_seq, maxlen=100)

prediction = model.predict(sample_pad)

if prediction[0][0] > 0.5:
    print("Spam")
else:
    print("Not Spam")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step
Not Spam
